# Structure Candidate Probe MSE Degradation

Train row/column/box candidate probes at the initial board state and evaluate AUC/MSE as the board advances.

In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from sudoku.probes.session import ProbeSession
from sudoku.probes.modes import cell_candidates
from sudoku.probes.activations import build_grid_at_step
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
LAYER = 4     # layer to probe (0-indexed)
N_CELLS = 81    # probe all cells
SEED = 42

## Load session

In [ ]:
session = ProbeSession(CACHE_PATH)

## Classify puzzles: simple vs hard

Simple = no PUSH/POP tokens anywhere in the sequence (pure constraint propagation).  
Hard = at least one PUSH token (backtracking was used).

In [ ]:
is_simple = np.array([
    not any(t == PUSH_TOKEN for t in seq)
    for seq in session.sequences
])

n_simple = is_simple.sum()
n_hard = (~is_simple).sum()
print(f"Simple puzzles (no backtracking): {n_simple:,}")
print(f"Hard puzzles (with backtracking): {n_hard:,}")

# Sequence length distribution check
seq_lens = np.array([len(s) for s in session.sequences])
print(f"\nSimple seq len: min={seq_lens[is_simple].min()}, max={seq_lens[is_simple].max()}, mean={seq_lens[is_simple].mean():.1f}")
print(f"Hard   seq len: min={seq_lens[~is_simple].min()}, max={seq_lens[~is_simple].max()}, mean={seq_lens[~is_simple].mean():.1f}")

## Build train/test split at step 0

- **Train**: all puzzles at step 0 (80% split)
- **Test**: the remaining 20%, further divided into simple/hard subsets

In [ ]:
idx0 = session.index.at_step(1).first_per_puzzle()
train_mask, test_mask = session.split(idx0, test_size=0.2, seed=SEED)

train_idx = idx0[train_mask]
test_idx  = idx0[test_mask]

# Identify which test puzzles are simple vs hard
test_puzzle_ids = test_idx.puzzle_idx
test_is_simple = is_simple[test_puzzle_ids]

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Test simple: {test_is_simple.sum():,}  |  Test hard: {(~test_is_simple).sum():,}")

## Helper functions

In [ ]:
from scripts.analysis.probe_experiments import (
    build_candidate_targets,
    fit_multilabel_probe as fit_cell_probe,
    eval_multilabel_probe as eval_cell_probe,
)


## Train probes at step 0

One multi-label LR probe per cell (81 probes total), trained on the training split at step 0.

In [ ]:
# Activations at step 0 for train and test splits
X_train_all = session.acts(train_idx, layer=LAYER)  # (n_train, d_model)
X_test_all  = session.acts(test_idx, layer=LAYER)   # (n_test,  d_model)

grids_train = session.grids(train_idx)
grids_test  = session.grids(test_idx)

print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

In [ ]:
# Train one probe per cell — filter to cells that have empty instances in train
cell_probes = {}  # cell_idx -> (clf, train_empty_mask)

for cell_idx in tqdm(range(N_CELLS), desc="Training probes"):
    targets_train, is_empty_train = build_candidate_targets(grids_train, cell_idx)
    if is_empty_train.sum() < 10:
        continue  # skip cells with almost no empty instances in train
    clf = fit_cell_probe(X_train_all[is_empty_train], targets_train[is_empty_train])
    cell_probes[cell_idx] = clf

print(f"Trained probes for {len(cell_probes)} cells")

In [ ]:
# Pre-identify test puzzle IDs for quick lookup
test_puzzle_id_set = set(test_puzzle_ids.tolist())
simple_test_puzzle_ids = set(test_puzzle_ids[test_is_simple].tolist())
hard_test_puzzle_ids   = set(test_puzzle_ids[~test_is_simple].tolist())


In [ ]:
from scripts.analysis.probe_experiments import eval_multilabel_probe as eval_structure_probe


In [ ]:
from scripts.analysis.probe_experiments import build_structure_targets, fit_multilabel_probe

# --- Train one probe per substructure at step 0 ---
SUBTYPES = (
    [('row', i) for i in range(9)]
    + [('col', i) for i in range(9)]
    + [('box', i) for i in range(9)]
)

struct_probes = {}  # (subtype, sidx) -> clf
for subtype, sidx in tqdm(SUBTYPES, desc='Training structure probes'):
    y_tr = build_structure_targets(grids_train, subtype, sidx)
    struct_probes[(subtype, sidx)] = fit_multilabel_probe(X_train_all, y_tr)

print(f'Trained {len(struct_probes)} structure probes')


In [ ]:
# Shared n_empty sweep setup used by structure-probe evaluation.
all_n_empty = 81 - session.index.n_filled
n_empty_values = sorted(int(x) for x in np.unique(all_n_empty) if 0 <= x <= 81)
print(f"n_empty range: {min(n_empty_values)}..{max(n_empty_values)} ({len(n_empty_values)} values)")


In [ ]:
struct_results = {
    'simple': {'auc': [], 'MSE': [], 'n': [], 'n_empty': []},
    'hard':   {'auc': [], 'MSE': [], 'n': [], 'n_empty': []},
}

for n_empty in tqdm(n_empty_values, desc='Evaluating structure probes'):
    n_filled_target = 81 - n_empty
    idx_slot = session.index.where_filled(n_filled_target).first_per_puzzle()
    in_test = np.isin(idx_slot.puzzle_idx, list(test_puzzle_id_set))
    idx_slot = idx_slot[in_test]

    if len(idx_slot) == 0:
        for grp in struct_results:
            struct_results[grp]['auc'].append(float('nan'))
            struct_results[grp]['MSE'].append(float('nan'))
            struct_results[grp]['n'].append(0)
            struct_results[grp]['n_empty'].append(n_empty)
        continue

    slot_is_simple = np.isin(idx_slot.puzzle_idx, list(simple_test_puzzle_ids))
    masks = {'simple': slot_is_simple, 'hard': ~slot_is_simple}

    X_slot = session.acts(idx_slot, layer=LAYER)
    grids_slot = session.grids(idx_slot)

    for grp, mask in masks.items():
        struct_results[grp]['n_empty'].append(n_empty)
        if mask.sum() == 0:
            struct_results[grp]['auc'].append(float('nan'))
            struct_results[grp]['MSE'].append(float('nan'))
            struct_results[grp]['n'].append(0)
            continue

        X_grp = X_slot[mask]
        grids_grp = [grids_slot[i] for i in np.where(mask)[0]]

        sub_aucs, sub_MSEs = [], []
        for (subtype, sidx), clf in struct_probes.items():
            y = build_structure_targets(grids_grp, subtype, sidx)
            auc, MSE = eval_structure_probe(clf, X_grp, y)
            if not np.isnan(auc):
                sub_aucs.append(auc)
            if not np.isnan(MSE):
                sub_MSEs.append(MSE)

        struct_results[grp]['auc'].append(float(np.mean(sub_aucs)) if sub_aucs else float('nan'))
        struct_results[grp]['MSE'].append(float(np.mean(sub_MSEs)) if sub_MSEs else float('nan'))
        struct_results[grp]['n'].append(int(mask.sum()))

print('Done.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {'simple': 'steelblue', 'hard': 'tomato'}
labels = {'simple': 'Simple (no backtracking)', 'hard': 'Hard (backtracking)'}

for grp in ['simple', 'hard']:
    xs     = struct_results[grp]['n_empty']
    aucs   = struct_results[grp]['auc']
    MSEs = struct_results[grp]['MSE']
    ns     = struct_results[grp]['n']

    valid_a = [(x, a) for x, a, n in zip(xs, aucs, ns) if not np.isnan(a) and n > 0]
    valid_b = [(x, b) for x, b, n in zip(xs, MSEs, ns) if not np.isnan(b) and n > 0]

    xa, ya = zip(*valid_a) if valid_a else ([], [])
    xb, yb = zip(*valid_b) if valid_b else ([], [])

    mean_n = int(np.mean([n for n in ns if n > 0])) if any(n > 0 for n in ns) else 0
    axes[0].plot(xa, ya, color=colors[grp], linewidth=1.8, marker='o', markersize=3,
                 label=f'{labels[grp]} (n≈{mean_n})')
    axes[1].plot(xb, yb, color=colors[grp], linewidth=1.8, marker='o', markersize=3)

for ax in axes:
    ax.set_xlabel('Number of empty cells in the grid')
    ax.invert_xaxis()
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Average AUC')
axes[0].set_title('AUC scores of probes evaluated at later grid states')
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

axes[1].set_ylabel('Average MSE')
axes[1].set_title('MSE scores of probes evaluated at later grid states')
axes[1].legend(
    handles=[plt.Line2D([0], [0], color=colors[grp], linewidth=2) for grp in ['simple', 'hard']],
    labels=[labels[grp] for grp in ['simple', 'hard']],
)

plt.tight_layout()
plt.savefig('MSE_degradation_structure_by_nempty.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved MSE_degradation_structure_by_nempty.png')
